# 03 — Modelling
Part of the *Kenya Household Food Security Classification* capstone project.

Majority-class baseline (Section 8), then model training: default XGBoost,
RandomizedSearchCV-tuned XGBoost, and CatBoost with native categorical handling,
followed by model selection on macro F1 (Section 9).

**Input:** `data/processed/02_preprocessed.pkl` (from `02_preprocessing.ipynb`)
**Output:** `data/processed/03_model.pkl` (consumed by `04_evaluation.ipynb`)


## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
path = "/content/drive/MyDrive/Ngao Labs Program/Capstone Project"

In [3]:
import pandas as pd
import numpy as np
import pickle
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

with open(path + '../data/processed/02_preprocessed.pkl', 'rb') as f:
    _S = pickle.load(f)
locals().update(_S)
print('Loaded preprocessing artifacts. X_train:', X_train.shape, ' X_test:', X_test.shape)


Loaded preprocessing artifacts. X_train: (3915, 68)  X_test: (979, 68)


## 8. Baseline

For multi-class classification, the baseline always predicts the majority class from
the training set, with no learning involved.

In [4]:
majority_class = stats.mode(y_train, keepdims=True)[0][0]
baseline_preds = np.full_like(y_test, majority_class)

baseline_weighted_f1 = f1_score(y_test, baseline_preds, average='weighted')
baseline_macro_f1 = f1_score(y_test, baseline_preds, average='macro')

print(f"Baseline (always predict '{le.classes_[majority_class]}'):")
print(f"  Weighted F1: {baseline_weighted_f1:.3f}")
print(f"  Macro F1:    {baseline_macro_f1:.3f}")

print("\nNote: with four classes now instead of three, the majority-class baseline's")
print("weighted F1 is structurally lower than before -- there is more room for a real")
print("model to add value, and macro F1 in particular should be watched closely since")
print("three of the four classes are now minority classes.")

Baseline (always predict 'food_secure'):
  Weighted F1: 0.333
  Macro F1:    0.167

Note: with four classes now instead of three, the majority-class baseline's
weighted F1 is structurally lower than before -- there is more room for a real
model to add value, and macro F1 in particular should be watched closely since
three of the four classes are now minority classes.


## 9. Model Training

In [5]:
from sklearn.utils.class_weight import compute_sample_weight

# Class-balanced sample weights: without this, XGBoost tends to over-predict the
# majority 'food_secure' class.
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    random_state=RANDOM_STATE
)
model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred = model.predict(X_test)

model_weighted_f1 = f1_score(y_test, y_pred, average='weighted')
model_macro_f1 = f1_score(y_test, y_pred, average='macro')
print(f"XGBoost (default + balanced weights): weighted F1={model_weighted_f1:.3f}, macro F1={model_macro_f1:.3f}")

XGBoost (default + balanced weights): weighted F1=0.538, macro F1=0.454


### 9.1 Hyperparameter Tuning (XGBoost)

`RandomizedSearchCV` over depth, learning rate, number of trees, and subsampling,
optimizing macro F1 (the metric most sensitive to how well we detect the three minority
insecure classes) with 5-fold stratified CV.

In [6]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

param_dist = {
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'n_estimators': [100, 200, 300, 500],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
}
base_est = XGBClassifier(objective='multi:softprob', num_class=len(le.classes_),
                          eval_metric='mlogloss', random_state=RANDOM_STATE)
search = RandomizedSearchCV(
    base_est, param_distributions=param_dist, n_iter=25, scoring='f1_macro',
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE, n_jobs=-1
)
search.fit(X_train, y_train, sample_weight=sample_weights)

print("Best params:", search.best_params_)
xgb_tuned = search.best_estimator_
y_pred_tuned = xgb_tuned.predict(X_test)
tuned_weighted_f1 = f1_score(y_test, y_pred_tuned, average='weighted')
tuned_macro_f1 = f1_score(y_test, y_pred_tuned, average='macro')
print(f"XGBoost (tuned): weighted F1={tuned_weighted_f1:.3f}, macro F1={tuned_macro_f1:.3f}")

Best params: {'subsample': 1.0, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 4, 'learning_rate': 0.05, 'colsample_bytree': 0.6}
XGBoost (tuned): weighted F1=0.531, macro F1=0.451


### 9.2 Alternative: CatBoost with native categorical handling

`head_gender`, `strata`, `head_age_3group`, `head_education_level`, `s2_q19_floormat`,
`s2_q20_wallmat`, and `s6_q1_soldasset` are one-hot encoded for XGBoost, which fragments
`s2_q20_wallmat`'s ~17 categories into that many sparse binary columns. CatBoost accepts
categorical columns directly — including `s2_q9a_county` in its raw, un-encoded
47-category form, sidestepping the target-encoding step entirely for this path.

In [7]:
!pip install catboost -q
from catboost import CatBoostClassifier, Pool

raw_cat_cols = cat_cols + ['s2_q9a_county']
raw_cols = raw_cat_cols + bin_cols + num_cols

X_raw = df[raw_cols].copy()
for c in raw_cat_cols:
    # .astype(str) alone leaves real NaNs as float NaN (a pandas quirk), which CatBoost
    # rejects in cat_features -- fillna first so missing values become an explicit category.
    X_raw[c] = X_raw[c].fillna('missing').astype(str)

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

cat_feature_idx = [X_raw.columns.get_loc(c) for c in raw_cat_cols]
train_pool = Pool(X_train_raw, y_train_raw, cat_features=cat_feature_idx)
test_pool = Pool(X_test_raw, cat_features=cat_feature_idx)

cb_model = CatBoostClassifier(
    iterations=400, depth=6, learning_rate=0.05,
    loss_function='MultiClass', auto_class_weights='Balanced',
    random_seed=RANDOM_STATE, verbose=False
)
cb_model.fit(train_pool)
y_pred_cb = cb_model.predict(test_pool).flatten().astype(int)

cb_weighted_f1 = f1_score(y_test_raw, y_pred_cb, average='weighted')
cb_macro_f1 = f1_score(y_test_raw, y_pred_cb, average='macro')
print(f"CatBoost (native categoricals): weighted F1={cb_weighted_f1:.3f}, macro F1={cb_macro_f1:.3f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.8 MB/s eta 0:00:00
CatBoost (native categoricals): weighted F1=0.518, macro F1=0.446


### 9.3 Model Selection

Compare all three candidates on the held-out test set and carry the best one (by macro
F1, tie-broken by weighted F1) forward into Evaluation and Feature Importance.

In [8]:
candidates = {
    'XGBoost (default + balanced weights)': {'model': model, 'X_test': X_test, 'y_pred': y_pred},
    'XGBoost (tuned via RandomizedSearchCV)': {'model': xgb_tuned, 'X_test': X_test, 'y_pred': y_pred_tuned},
    'CatBoost (native categoricals)': {'model': cb_model, 'X_test': X_test_raw, 'y_pred': y_pred_cb},
}

comparison_rows = []
for name, c in candidates.items():
    wf1 = f1_score(y_test, c['y_pred'], average='weighted')
    mf1 = f1_score(y_test, c['y_pred'], average='macro')
    comparison_rows.append({'model': name, 'weighted_f1': round(wf1, 3), 'macro_f1': round(mf1, 3)})

comparison = pd.DataFrame(comparison_rows).sort_values('macro_f1', ascending=False).reset_index(drop=True)
print(comparison.to_string(index=False))

best_name = comparison.iloc[0]['model']
best_candidate = candidates[best_name]
final_model = best_candidate['model']
final_X_test = best_candidate['X_test']
final_y_pred = best_candidate['y_pred']
print(f"\nBest model by macro F1: {best_name}")

                                 model  weighted_f1  macro_f1
  XGBoost (default + balanced weights)        0.538     0.454
XGBoost (tuned via RandomizedSearchCV)        0.531     0.451
        CatBoost (native categoricals)        0.518     0.446

Best model by macro F1: XGBoost (default + balanced weights)


## 15. Save Output for the Next Notebook

In [11]:
import os

output_dir = os.path.join(path, 'data', 'processed')
os.makedirs(output_dir, exist_ok=True)

_forward = dict(_S)  # carry everything from 02_preprocessed.pkl forward too
_forward.update({
    'sample_weights': sample_weights,
    'model': model, 'y_pred': y_pred,
    'xgb_tuned': xgb_tuned, 'y_pred_tuned': y_pred_tuned,
    'cb_model': cb_model, 'y_pred_cb': y_pred_cb,
    'X_train_raw': X_train_raw, 'X_test_raw': X_test_raw,
    'y_train_raw': y_train_raw, 'y_test_raw': y_test_raw,
    'raw_cat_cols': raw_cat_cols, 'cat_feature_idx': cat_feature_idx,
    'comparison': comparison, 'best_name': best_name,
    'final_model': final_model, 'final_X_test': final_X_test, 'final_y_pred': final_y_pred,
})

output_file_path = os.path.join(output_dir, '03_model.pkl')
with open(output_file_path, 'wb') as f:
    pickle.dump(_forward, f)

print(f'Saved modelling artifacts to {output_file_path}')
print('Winning model:', best_name)

Saved modelling artifacts to /content/drive/MyDrive/Ngao Labs Program/Capstone Project/data/processed/03_model.pkl
Winning model: XGBoost (default + balanced weights)
